In [2]:
import math
import torch as t
import torch.nn as nn
import torch.nn.functional as F

In [3]:
class SelfAttention(nn.Module):
    """
    (B, L, E) -> (B, L, E)
    """
    def __init__(self, ctx_len, dim, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.dim = dim
        self.Qw = nn.Linear(dim, dim)
        self.Kw = nn.Linear(dim, dim)
        self.Vw = nn.Linear(dim, dim)

    def forward(self, x, mask=None):

        # (B, L, E) -> (B, L, E)

        Q = self.Qw(x)
        K = self.Kw(x)
        V = self.Vw(x)
        
        dot_prod = Q @ K.permute(0, 2, 1)  # (B, L, E) x (B, E, L) = (B, L , L)

        # size = (B, L, L)
        scaled_dot = dot_prod / math.sqrt(self.dim)

        # 3. Apply the Mask
        if mask is not None:
            # .masked_fill takes a condition, and replaces values with what you tell it.
            # Assuming mask has 0s where we want to hide stuff, and 1s where it's safe to look.
            masked_scaled_dot = scaled_dot.masked_fill(mask == 0, float('-inf')) 

        attention_weights = F.softmax(masked_scaled_dot, dim=-1)

        # (B, L, L) x (B, L, E) = (B, L, E)
        out = attention_weights @ V

        return out
        

In [4]:
class MultiHeadedAttention(nn.Module):
    """
    (B, L, E) -> (B, L, E)
    """
    def __init__(self, heads, ctx_len, dim, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert dim % heads == 0, "dim must be divisible by n_heads"

        self.dim = dim
        self.heads = heads # no.of attention heads
        self.head_dim = dim//heads

        self.Qw = nn.Linear(dim, dim)
        self.Kw = nn.Linear(dim, dim)
        self.Vw = nn.Linear(dim, dim)
        self.Wo = nn.Linear(dim, dim) # output projection. 

        causal_mask = t.tril(t.ones(ctx_len, ctx_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x: t.Tensor):

        # (B, L, E) -> (B, L, E)
        B, L, _ = x.size()

        Q = self.Qw(x).view(B, L, self.heads, self.head_dim).permute(0, 2, 1, 3)
        K = self.Kw(x).view(B, L, self.heads, self.head_dim).permute(0, 2, 1, 3)
        V = self.Vw(x).view(B, L, self.heads, self.head_dim).permute(0, 2, 1, 3)
        
        dot_prod = Q @ K.permute(0, 1, 3, 2)  # (B, H, L, head_dim) x (B, H, head_dim, L) = (B, H, L, L)
           
        # size = (B, n, L, L)
        scaled_dot = dot_prod / math.sqrt(self.head_dim)

        # applying causal mask. 
        causal = self.causal_mask[:L, :L]  # slicing to the current length (only affects the edge cases).
        masked_scaled_dot = scaled_dot.masked_fill(causal == 0, float("-inf"))

        attention_weights = F.softmax(masked_scaled_dot, dim=-1)

        # (B, n, L, L) x (B, n, L, E/n) = (B, n, L, E/n)
        out = attention_weights @ V 

        # we first permute it, and then join emb_dims from diff heads together. 
        out = out.permute(0, 2, 1, 3).contiguous().view(B, L, self.dim) # (B, L, E)
        out = self.Wo(out)
        return out

In [16]:
att = MultiHeadedAttention(8, 1024, 256)

input = t.rand(4, 1024, 256)

interesting = att(input)

In [17]:
print(interesting)

tensor([[[ 0.1354,  0.2412,  0.1957,  ..., -0.1162,  0.2042, -0.3605],
         [ 0.1047,  0.1901,  0.1907,  ..., -0.0453,  0.1519, -0.3164],
         [ 0.1284,  0.1938,  0.1602,  ..., -0.0373,  0.1408, -0.2987],
         ...,
         [ 0.0891,  0.1562,  0.1417,  ..., -0.1264, -0.0111, -0.3257],
         [ 0.0892,  0.1562,  0.1411,  ..., -0.1263, -0.0109, -0.3254],
         [ 0.0888,  0.1563,  0.1413,  ..., -0.1267, -0.0112, -0.3257]],

        [[ 0.0801,  0.0734,  0.0796,  ..., -0.0857, -0.0247, -0.5435],
         [ 0.0793,  0.1235,  0.1491,  ..., -0.0843, -0.0327, -0.4455],
         [ 0.0093,  0.1602,  0.1628,  ..., -0.0898, -0.0136, -0.3774],
         ...,
         [ 0.0878,  0.1549,  0.1435,  ..., -0.1304, -0.0112, -0.3283],
         [ 0.0881,  0.1551,  0.1432,  ..., -0.1306, -0.0110, -0.3283],
         [ 0.0878,  0.1549,  0.1432,  ..., -0.1306, -0.0110, -0.3283]],

        [[ 0.1013, -0.0109,  0.0534,  ..., -0.0817, -0.0812, -0.5443],
         [ 0.1258,  0.0808,  0.0382,  ..., -0

In [10]:
class MLP(nn.Module):
    def __init__(self, dim, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.fc1 = nn.Linear(dim, 4*dim)
        self.fc2 = nn.Linear(4*dim, dim)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)  # gaussian error linear unit.
        x = self.fc2(x)
        return x

In [14]:
class TransformerBlock(nn.Module):
    def __init__(self, ctx_len: int, dim: int, Attention: nn.Module, heads: int = 8):
        super().__init__()

        self.layer_norm1 = nn.LayerNorm(dim)
        self.layer_norm2 = nn.LayerNorm(dim)
        self.attention = Attention(heads, ctx_len, dim)

        self.mlp = MLP(dim)



    def forward(self, x):
        residue = x 
        
        x = self.layer_norm1(x)
        x = self.attention(x)
        
        x += residue

        residue = x

        x = self.layer_norm2(x)
        x = self.mlp(x)

        x += residue

        return x

In [18]:
trans = TransformerBlock(1024, 256, MultiHeadedAttention, 8)
output = trans(input)

In [20]:
output.size()

torch.Size([4, 1024, 256])

In [ ]:
class ModularTransformer(nn.Module):
    def __init__(self, ctx_len: int, dim: int, attention: nn.Module, transformer_block: nn.Module, positional_embedding: nn.Module, heads: int, vocab_size: int, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.PE = positional_embedding(dim, ctx_len)
        self.transformer_block = transformer_block(ctx_len, dim, attention, heads)
        self.fc = nn.Linear(dim, vocab_size)
        self.softmax = nn.Softmax()

    
    def forward(self, x):
        x = self.PE(x)
        x = self.transformer_block(x)
        x = self.softmax(self.fc(x))

        return x